# CAA2026 Data Preparation

This notebook generates a complete dataset for archaeological site detection using:
- **Sentinel-2** satellite imagery (10m resolution)
- **FABDEM** digital elevation model
- **Position sampling** to avoid center bias
- **Rotation augmentation** for geometric invariance
- **Multiple label variants** (hard + soft with different sigmas)
- **Zarr storage on GCS** for efficient cloud-native access

## 1. Setup & Configuration

In [ ]:
!pip install earthengine-api pandas numpy scipy matplotlib pyyaml pyarrow shapely rasterio affine pyproj tqdm gcsfs zarr -q

In [ ]:
# ============================================================================
# MODIFY THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================================

# ----------------------------------------------------------------------------
# Google Drive Paths
# ----------------------------------------------------------------------------

# Your Google Drive working directory (where config/ folder and input files are located)
# Example: '/content/drive/My Drive/MyProject' or '/content/drive/MyDrive/CAA2026'
WORKING_DIR = 'YOUR-GOOGLE-DRIVE-WORKING-DIRECTORY'

# Path to your settings YAML file (relative to WORKING_DIR)
# Usually: 'config/settings.yaml' if you follow the standard structure
CONFIG_FILE = 'config/settings.yaml'


# ----------------------------------------------------------------------------
# Google Cloud Platform Settings
# ----------------------------------------------------------------------------

# Your GCP project ID for Earth Engine
# Get this from: https://console.cloud.google.com/
GEE_PROJECT_ID = 'YOUR-GCP-PROJECT-ID'

# Your GCS bucket name (without gs:// prefix)
# Example: 'my-archaeological-data' (not 'gs://my-archaeological-data')
GCS_BUCKET_NAME = 'YOUR-GCS-BUCKET-NAME'


# ----------------------------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ----------------------------------------------------------------------------

print("=" * 70)
print("USER CONFIGURATION LOADED")
print("=" * 70)
print(f"Working directory:  {WORKING_DIR}")
print(f"Config file:        {CONFIG_FILE}")
print(f"GEE Project:        {GEE_PROJECT_ID}")
print(f"GCS Bucket:         {GCS_BUCKET_NAME}")
print("=" * 70)
print("\n⚠️  If you see 'YOUR-...' values above, you need to configure this block!")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# Import Libraries
# ----------------------------------------------------------------------------

# Core libraries
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import json
import math
import zarr
import gcsfs
import os
from pathlib import Path
from datetime import datetime, timezone
from tqdm.notebook import tqdm
import random
import gc
import time
from zarr.codecs import BloscCodec

# Scipy
from scipy.ndimage import rotate, distance_transform_edt, zoom

# Geospatial
from affine import Affine
from rasterio.features import rasterize
from shapely.geometry import shape, box
from shapely.ops import transform as shapely_transform
from shapely.affinity import rotate as shapely_rotate
from shapely.strtree import STRtree
from pyproj import CRS, Transformer

# Colab-specific
from google.colab import drive, auth


# ----------------------------------------------------------------------------
# Mount Drive & Set Working Directory
# ----------------------------------------------------------------------------

drive.mount('/content/drive')
os.chdir(WORKING_DIR)  # Use user-configured path


# ----------------------------------------------------------------------------
# Authentication
# ----------------------------------------------------------------------------

# Google Cloud Storage
auth.authenticate_user()

# Google Earth Engine
try:
    ee.Initialize(project=GEE_PROJECT_ID)  # Use user-configured project
except:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)


# ----------------------------------------------------------------------------
# Load Configuration & Define Constants
# ----------------------------------------------------------------------------

def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

# Load configuration using user-specified path
config = load_config(CONFIG_FILE)

# GCS paths using user-configured bucket
DATASET_PATH = f"gs://{GCS_BUCKET_NAME}/dataset/"

# Channel order constant (used throughout pipeline)
CHANNEL_ORDER = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'DEM', 'Slope', 'NDVI', 'NDWI', 'BSI']


# ----------------------------------------------------------------------------
# Display Configuration Summary
# ----------------------------------------------------------------------------

print("=" * 70)
print("SETUP COMPLETE")
print("=" * 70)
print(f"✓ Working directory:  {os.getcwd()}")
print(f"✓ GCS Bucket:         {GCS_BUCKET_NAME}")
print(f"✓ Dataset path:       {DATASET_PATH}")
print(f"✓ GEE Project:        {GEE_PROJECT_ID}")
print(f"✓ Grid size:          {config['grid']['cell_size_km']} km × {config['grid']['pixels_per_km']} pixels")
print(f"✓ Position sampling:  {config['position_sampling']['enabled']}")
print(f"✓ Rotation enabled:   {config['augmentation']['rotation_generation']['enabled']}")
print(f"✓ Rotation step:      {config['augmentation']['rotation_generation']['rotation_step']}°")
print(f"✓ Input sites:        {config['paths']['input_sites']}")
print(f"✓ Channels:           {len(CHANNEL_ORDER)} total")
print("=" * 70)

## 2. Helper Classes

In [ ]:
# ----------------------------------------------------------------------------
# 2.1 GEEExtractor - Extract Satellite Imagery and Elevation Data
# ----------------------------------------------------------------------------

class GEEExtractor:
    """
    Extract multi-spectral satellite imagery and elevation data from Google Earth Engine.

    This class handles:
    - Sentinel-2 imagery extraction (optical bands)
    - FABDEM elevation data extraction
    - Slope calculation from DEM
    - Automatic cloud filtering and image selection

    All extractions are performed at configurable resolution and spatial extent.
    """

    def __init__(self, config):
        """
        Initialize extractor with configuration.

        Args:
            config: Configuration dictionary containing GEE and grid settings
        """
        self.config = config
        self.s2_config = config['imagery']['sentinel2']
        self.fabdem_config = config['imagery']['fabdem']
        self.grid_config = config['grid']

    def create_grid_bbox(self, lat, lon):
        """
        Create a bounding box centered at given coordinates.

        Args:
            lat: Latitude of center point
            lon: Longitude of center point

        Returns:
            ee.Geometry.Rectangle: Bounding box for the grid cell
        """
        cell_size_km = self.grid_config['cell_size_km']
        half_size_deg = (cell_size_km / 2) / 111.32  # Convert km to degrees

        min_lon = lon - half_size_deg
        max_lon = lon + half_size_deg
        min_lat = lat - half_size_deg
        max_lat = lat + half_size_deg

        return ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

    def get_sentinel2_image(self, roi):
        """
        Get the best Sentinel-2 image for the region of interest.

        Selects the image with lowest cloud cover within the configured date range.

        Args:
            roi: ee.Geometry defining the region of interest

        Returns:
            ee.Image: Best Sentinel-2 image for the region
        """
        collection = (
            ee.ImageCollection(self.s2_config['collection'])
            .filterBounds(roi)
            .filterDate(self.s2_config['date_start'], self.s2_config['date_end'])
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', self.s2_config['cloud_cover_max']))
            .sort('CLOUDY_PIXEL_PERCENTAGE')  # Best image first
        )

        image = ee.Image(collection.first())
        return image

    def get_fabdem_data(self, roi):
        """
        Get FABDEM elevation data for the region.

        Args:
            roi: ee.Geometry defining the region of interest

        Returns:
            ee.Image: FABDEM elevation data
        """
        fabdem = (ee.ImageCollection(self.fabdem_config['collection'])
                  .filterBounds(roi)
                  .first())
        return fabdem

    def calculate_slope(self, dem_image):
        """
        Calculate slope from DEM using Earth Engine's terrain algorithm.

        Args:
            dem_image: ee.Image containing elevation data

        Returns:
            ee.Image: Slope in degrees
        """
        slope = ee.Terrain.slope(dem_image)
        return slope

    def extract_band_array(self, image, band, roi):
        """
        Extract a single band as a NumPy array at configured resolution.

        Handles resampling if the extracted size doesn't match target size.

        Args:
            image: ee.Image to extract from
            band: Band name to extract
            roi: ee.Geometry defining extraction region

        Returns:
            np.ndarray: 2D array of pixel values (float32)
        """
        pixels = self.grid_config['pixels_per_km']
        scale = self.s2_config['scale']

        band_image = image.select(band)

        # Extract as array
        array = band_image.sampleRectangle(region=roi, defaultValue=0)
        data = array.get(band).getInfo()

        arr = np.array(data, dtype=np.float32)

        # Resample if needed (handles edge cases where size doesn't match exactly)
        if arr.shape != (pixels, pixels):
            zoom_factors = (pixels / arr.shape[0], pixels / arr.shape[1])
            arr = zoom(arr, zoom_factors, order=1)

        return arr

    def extract_all_channels(self, lat, lon):
        """
        Extract all spectral bands, DEM, and slope for a location.

        This is the main method that orchestrates the complete extraction.

        Args:
            lat: Latitude of extraction center
            lon: Longitude of extraction center

        Returns:
            tuple: (channels_dict, metadata_dict)
                - channels_dict: Dictionary mapping channel names to 2D arrays
                - metadata_dict: Image metadata (ID, cloud cover, acquisition date)
        """
        # Create bounding box
        roi = self.create_grid_bbox(lat, lon)

        # Get imagery
        s2_image = self.get_sentinel2_image(roi)
        fabdem_image = self.get_fabdem_data(roi)
        slope_image = self.calculate_slope(fabdem_image)

        # Extract all channels
        channels = {}

        # Sentinel-2 bands
        for band in self.s2_config['bands']:
            channels[band] = self.extract_band_array(s2_image, band, roi)

        # Elevation and slope
        channels['DEM'] = self.extract_band_array(fabdem_image, 'b1', roi)
        channels['Slope'] = self.extract_band_array(slope_image, 'slope', roi)

        # Extract metadata
        image_info = s2_image.getInfo()
        metadata = {
            'image_id': image_info['id'] if image_info else None,
            'cloud_cover': image_info['properties'].get('CLOUDY_PIXEL_PERCENTAGE') if image_info else None,
            'acquisition_date': image_info['properties'].get('GENERATION_TIME') if image_info else None
        }

        return channels, metadata

print("✓ GEEExtractor class defined")

In [ ]:
# ----------------------------------------------------------------------------
# 2.2 IndexCalculator - Compute Spectral Indices
# ----------------------------------------------------------------------------

class IndexCalculator:
    """
    Calculate spectral indices from Sentinel-2 bands.

    Spectral indices are mathematical combinations of bands that highlight
    specific land cover features:
    - NDVI: Vegetation health and density
    - NDWI: Water content and moisture
    - BSI: Bare soil and exposed ground

    All calculations handle division by zero safely.
    """

    @staticmethod
    def calculate_ndvi(b8, b4):
        """
        Calculate Normalized Difference Vegetation Index.

        NDVI = (NIR - Red) / (NIR + Red)

        Values range from -1 to 1:
        - High values (0.6-0.9): Dense vegetation
        - Medium values (0.2-0.5): Sparse vegetation
        - Low values (<0.2): Bare soil, rock, water

        Args:
            b8: Near-Infrared band (NIR)
            b4: Red band

        Returns:
            np.ndarray: NDVI values (float32)
        """
        numerator = b8 - b4
        denominator = b8 + b4

        # Safe division: returns 0 where denominator is 0
        ndvi = np.divide(numerator, denominator,
                        out=np.zeros_like(numerator),
                        where=denominator!=0)

        return ndvi.astype(np.float32)

    @staticmethod
    def calculate_ndwi(b3, b8):
        """
        Calculate Normalized Difference Water Index.

        NDWI = (Green - NIR) / (Green + NIR)

        Values range from -1 to 1:
        - High values (>0.3): Water bodies
        - Medium values (0-0.3): Wet soil, vegetation moisture
        - Low values (<0): Dry areas

        Args:
            b3: Green band
            b8: Near-Infrared band (NIR)

        Returns:
            np.ndarray: NDWI values (float32)
        """
        numerator = b3 - b8
        denominator = b3 + b8

        # Safe division: returns 0 where denominator is 0
        ndwi = np.divide(numerator, denominator,
                        out=np.zeros_like(numerator),
                        where=denominator!=0)

        return ndwi.astype(np.float32)

    @staticmethod
    def calculate_bsi(b11, b4, b8, b2):
        """
        Calculate Bare Soil Index.

        BSI = ((SWIR + Red) - (NIR + Blue)) / ((SWIR + Red) + (NIR + Blue))

        Values range from -1 to 1:
        - High values (>0.1): Bare soil, exposed ground
        - Medium values (-0.1 to 0.1): Mixed surfaces
        - Low values (<-0.1): Vegetation, water

        Useful for detecting archaeological features that disturb soil.

        Args:
            b11: SWIR band (Shortwave Infrared)
            b4: Red band
            b8: Near-Infrared band (NIR)
            b2: Blue band

        Returns:
            np.ndarray: BSI values (float32)
        """
        numerator = (b11 + b4) - (b8 + b2)
        denominator = (b11 + b4) + (b8 + b2)

        # Safe division: returns 0 where denominator is 0
        bsi = np.divide(numerator, denominator,
                       out=np.zeros_like(numerator),
                       where=denominator!=0)

        return bsi.astype(np.float32)

    @staticmethod
    def calculate_all_indices(channels):
        """
        Calculate all spectral indices from a channels dictionary.

        This is a convenience method that computes NDVI, NDWI, and BSI
        in one call, using the standard Sentinel-2 band names.

        Args:
            channels: Dictionary with keys 'B2', 'B3', 'B4', 'B8', 'B11'

        Returns:
            dict: Dictionary with keys 'NDVI', 'NDWI', 'BSI' mapping to arrays
        """
        indices = {}

        # Vegetation index
        indices['NDVI'] = IndexCalculator.calculate_ndvi(
            channels['B8'], channels['B4']
        )

        # Water index
        indices['NDWI'] = IndexCalculator.calculate_ndwi(
            channels['B3'], channels['B8']
        )

        # Bare soil index
        indices['BSI'] = IndexCalculator.calculate_bsi(
            channels['B11'], channels['B4'], channels['B8'], channels['B2']
        )

        return indices

print("✓ IndexCalculator class defined")

In [ ]:
# ----------------------------------------------------------------------------
# 2.3 LabelGenerator - Generate Ground Truth Labels
# ----------------------------------------------------------------------------

class LabelGenerator:
    """
    Generate multiple label variants from archaeological contour geometries.

    This class creates both hard (binary) and soft (probabilistic) labels:
    - Hard labels: Binary masks (0 or 1) marking exact contour locations
    - Soft labels: Gaussian-smoothed probabilities with configurable sigma values

    Soft labels with different sigma values allow the model to learn features
    at different spatial scales around archaeological sites.
    """

    def __init__(self, config, contours_geojson_path):
        """
        Initialize label generator with configuration and contour data.

        Args:
            config: Configuration dictionary with label generation settings
            contours_geojson_path: Path to GeoJSON file containing contour geometries
        """
        self.config = config
        self.label_config = config['label_generation']

        # Load contour geometries from GeoJSON
        self.contours_geojson_path = Path(contours_geojson_path)
        if not self.contours_geojson_path.exists():
            raise FileNotFoundError(f"Contours GeoJSON not found: {contours_geojson_path}")

        self.geoms_wgs84 = self._load_geojson_geoms()
        # Build spatial index for fast geometry queries
        self.strtree = STRtree(self.geoms_wgs84) if self.geoms_wgs84 else None

        # Label parameters
        self.size_px = self.label_config['size_px']
        self.tile_size_m = self.label_config['tile_size_m']
        self.query_radius_m = self.label_config['query_radius_m']

    def _load_geojson_geoms(self):
        """
        Load geometries from GeoJSON file.

        Returns:
            list: List of Shapely geometry objects
        """
        data = json.loads(self.contours_geojson_path.read_text(encoding='utf-8'))
        feats = data.get('features', [])
        geoms = []
        for f in feats:
            g = f.get('geometry', None)
            if not g:
                continue
            try:
                geoms.append(shape(g))
            except Exception:
                continue
        return geoms

    def _aeqd_crs(self, lat, lon):
        """
        Create Azimuthal Equidistant projection centered at a point.

        This projection preserves distances from the center point, which is
        essential for accurate spatial operations in meters.

        Args:
            lat: Center latitude
            lon: Center longitude

        Returns:
            CRS: Pyproj CRS object for the projection
        """
        proj4 = f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m +no_defs"
        return CRS.from_proj4(proj4)

    def _rasterize_geoms(self, geoms, all_touched=True):
        """
        Convert vector geometries to raster mask.

        Args:
            geoms: List of Shapely geometries in local metric coordinates
            all_touched: If True, all pixels touched by geometry are marked.
                        If False, only pixels whose center is inside are marked.

        Returns:
            np.ndarray: Binary mask (uint8) with 1 where geometries exist
        """
        if not geoms:
            return np.zeros((self.size_px, self.size_px), dtype=np.uint8)

        # Calculate pixel size in meters
        px = self.tile_size_m / float(self.size_px)

        # Create affine transform: origin at tile center
        transform = Affine(px, 0, -self.tile_size_m / 2.0,
                          0, -px, self.tile_size_m / 2.0)

        # Prepare shapes for rasterization
        shapes = [(g, 1) for g in geoms if (g is not None and not g.is_empty)]
        if not shapes:
            return np.zeros((self.size_px, self.size_px), dtype=np.uint8)

        # Rasterize geometries
        return rasterize(
            shapes=shapes,
            out_shape=(self.size_px, self.size_px),
            transform=transform,
            fill=0,
            all_touched=all_touched,
            dtype=np.uint8
        )

    def _compute_soft_labels(self, boundary_mask, sigmas=[1, 3, 8]):
        """
        Generate soft (probabilistic) labels with multiple sigma values.

        Uses distance transform and Gaussian decay to create smooth probability
        distributions around hard boundaries. Different sigma values capture
        features at different spatial scales:
        - Small sigma (1): Tight focus on exact boundaries
        - Medium sigma (3): Moderate spatial extent
        - Large sigma (8): Broad spatial context

        Args:
            boundary_mask: Binary mask (0/1) marking boundaries
            sigmas: List of sigma values for Gaussian smoothing

        Returns:
            dict: Maps 'sigma{N}' to probability arrays (float16)
        """
        # Compute distance from each pixel to nearest boundary
        # Distance is 0 at boundaries, increases elsewhere
        dist = distance_transform_edt(boundary_mask == 0).astype(np.float32)

        soft_labels = {}
        for sigma in sigmas:
            # Gaussian decay: exp(-(dist^2) / (2*sigma^2))
            # Pixels far from boundaries get lower probability
            prob = np.exp(-(dist ** 2) / (2.0 * (float(sigma) ** 2)))

            # Ensure boundary pixels are exactly 1.0
            prob[boundary_mask > 0] = 1.0

            # Store as float16 to save memory (sufficient precision for probabilities)
            soft_labels[f'sigma{sigma}'] = prob.astype(np.float16)

        return soft_labels

    def generate_positive_labels(self, lat, lon, rotation_angle):
        """
        Generate hard + soft labels for a positive sample (contains archaeological features).

        This method:
        1. Queries nearby contour geometries from the spatial index
        2. Projects them to local metric coordinates
        3. Applies rotation to match the rotated imagery
        4. Clips to the tile boundary
        5. Rasterizes to create hard labels
        6. Computes soft labels with multiple sigma values

        Args:
            lat: Center latitude
            lon: Center longitude
            rotation_angle: Rotation angle in degrees (0-360)

        Returns:
            dict: Contains 'hard' (uint8) and 'sigma1', 'sigma3', 'sigma8' (float16) arrays
        """
        # Define tile boundary in local coordinates (meters)
        tile = box(-self.tile_size_m / 2.0, -self.tile_size_m / 2.0,
                   self.tile_size_m / 2.0, self.tile_size_m / 2.0)

        # Create projection: WGS84 -> local metric coordinates
        local_crs = self._aeqd_crs(lat, lon)
        fwd = Transformer.from_crs(CRS.from_epsg(4326), local_crs, always_xy=True).transform

        # Create query bounding box in WGS84 (degrees)
        # This is larger than the tile to catch geometries that might intersect
        lat_rad = math.radians(lat)
        meters_per_deg_lat = 111_320.0
        meters_per_deg_lon = 111_320.0 * max(1e-6, math.cos(lat_rad))
        dlat = self.query_radius_m / meters_per_deg_lat
        dlon = self.query_radius_m / meters_per_deg_lon

        query_bbox = box(lon - dlon, lat - dlat, lon + dlon, lat + dlat)

        # Query spatial index for candidate geometries
        candidates = []
        if self.strtree:
            try:
                hits = self.strtree.query(query_bbox)
                # Handle different STRtree return formats
                if len(hits) > 0 and isinstance(hits[0], (int, np.integer)):
                    candidates = [self.geoms_wgs84[int(i)] for i in hits]
                else:
                    candidates = list(hits)
            except Exception:
                # Fallback: use all geometries if spatial index fails
                candidates = self.geoms_wgs84
        else:
            candidates = self.geoms_wgs84

        # Process geometries: project, rotate, clip
        line_geoms = []
        for g in candidates:
            # Quick check: does it intersect query box?
            if not g.intersects(query_bbox):
                continue

            # Project to local metric coordinates
            gm = shapely_transform(fwd, g)

            # Rotate around tile center
            gm = shapely_rotate(gm, rotation_angle, origin=(0, 0))

            # Clip to tile boundary
            clipped = gm.intersection(tile)
            if clipped.is_empty:
                continue

            line_geoms.append(clipped)

        # Generate hard label (binary mask)
        boundary_mask = self._rasterize_geoms(line_geoms, all_touched=True)

        # Generate all soft label variants from single distance transform
        soft_labels = self._compute_soft_labels(boundary_mask, sigmas=[1, 3, 8])

        return {
            'hard': boundary_mask,
            **soft_labels
        }

    def generate_negative_labels(self):
        """
        Generate zero labels for negative samples (no archaeological features).

        Returns:
            np.ndarray: All-zero array (uint8)
        """
        return np.zeros((self.size_px, self.size_px), dtype=np.uint8)

print("✓ LabelGenerator class defined")

In [ ]:
# ----------------------------------------------------------------------------
# 2.4 ProgressTracker - Resume-able Processing with GCS State
# ----------------------------------------------------------------------------

class ProgressTracker:
    """
    Track dataset generation progress using a JSON file stored on GCS.

    This enables:
    - Resuming interrupted processing runs
    - Skipping already-completed batches
    - Tracking which parts of the dataset are complete
    - Atomic updates to prevent corruption

    The progress file is stored on GCS so it persists across Colab sessions
    and can be shared across multiple processing instances.
    """

    def __init__(self, gcs_path):
        """
        Initialize progress tracker.

        Args:
            gcs_path: Full GCS path to progress file (e.g., 'gs://bucket/progress.json')
        """
        self.gcs_path = gcs_path
        self.fs = gcsfs.GCSFileSystem()
        self.progress = self._load()

    def _load(self):
        """
        Load progress from GCS, or create new progress structure if not found.

        Returns:
            dict: Progress data with completed batch lists and timestamp
        """
        try:
            with self.fs.open(self.gcs_path, 'r') as f:
                return json.load(f)
        except FileNotFoundError:
            # Initialize new progress tracking structure
            return {
                'completed_site_batches': [],
                'completed_landcover_batches': [],
                'last_updated': None
            }

    def _save(self):
        """
        Save progress to GCS with atomic write operation.

        Uses a temporary file + rename pattern to ensure atomic updates
        and prevent corruption from interrupted writes.
        """
        # Update timestamp
        self.progress['last_updated'] = datetime.now(timezone.utc).isoformat()

        # Write to temporary file first
        temp_path = self.gcs_path + '.tmp'
        with self.fs.open(temp_path, 'w') as f:
            json.dump(self.progress, f, indent=2)

        # Atomic rename (GCS guarantees atomicity)
        self.fs.mv(temp_path, self.gcs_path, overwrite=True)

    def is_site_batch_complete(self, batch_id):
        """
        Check if a site batch has already been processed.

        Args:
            batch_id: Batch identifier (integer)

        Returns:
            bool: True if batch is complete, False otherwise
        """
        return batch_id in self.progress['completed_site_batches']

    def is_landcover_batch_complete(self, batch_id):
        """
        Check if a landcover batch has already been processed.

        Args:
            batch_id: Batch identifier (integer)

        Returns:
            bool: True if batch is complete, False otherwise
        """
        return batch_id in self.progress['completed_landcover_batches']

    def mark_site_batch_complete(self, batch_id):
        """
        Mark a site batch as complete and persist to GCS.

        Keeps the completed batch list sorted for easier inspection.

        Args:
            batch_id: Batch identifier to mark complete
        """
        if batch_id not in self.progress['completed_site_batches']:
            self.progress['completed_site_batches'].append(batch_id)
            self.progress['completed_site_batches'].sort()
            self._save()

    def mark_landcover_batch_complete(self, batch_id):
        """
        Mark a landcover batch as complete and persist to GCS.

        Keeps the completed batch list sorted for easier inspection.

        Args:
            batch_id: Batch identifier to mark complete
        """
        if batch_id not in self.progress['completed_landcover_batches']:
            self.progress['completed_landcover_batches'].append(batch_id)
            self.progress['completed_landcover_batches'].sort()
            self._save()

    def get_summary(self):
        """
        Get a summary of current progress.

        Returns:
            dict: Contains counts of completed batches and last update timestamp
        """
        return {
            'site_batches_completed': len(self.progress['completed_site_batches']),
            'landcover_batches_completed': len(self.progress['completed_landcover_batches']),
            'last_updated': self.progress['last_updated']
        }

print("✓ ProgressTracker class defined")

## 3. Utility Functions

In [ ]:
# ----------------------------------------------------------------------------
# Position and Geometry Functions
# ----------------------------------------------------------------------------

def calculate_site_radius(row):
    """
    Calculate effective site radius from width measurements.

    Archaeological sites often have elliptical shapes defined by two axes
    (a_width and b_width). This function computes a conservative radius
    that encompasses the site.

    Args:
        row: DataFrame row with 'a_width' and 'b_width' columns

    Returns:
        float: Site radius in meters. Defaults to 250m if no width data.
    """
    a_width = row.get('a_width')
    b_width = row.get('b_width')

    # Try to convert to float, handle invalid/missing values
    try:
        a_width = float(a_width) if pd.notna(a_width) else None
    except (ValueError, TypeError):
        a_width = None

    try:
        b_width = float(b_width) if pd.notna(b_width) else None
    except (ValueError, TypeError):
        b_width = None

    # Calculate radius based on available data
    if a_width is None:
        return 250.0  # Default radius when no data available
    elif b_width is None:
        return a_width / 2.0  # Use half of single width
    else:
        return max(a_width, b_width) / 2.0  # Use larger dimension


def generate_position_offsets(site_radius_m, config):
    """
    Generate position offsets to avoid center bias in training data.

    Instead of always placing the site at the tile center, this function
    generates random positions within safe bounds. This prevents the model
    from learning that sites are always centered.

    Args:
        site_radius_m: Site radius in meters
        config: Configuration dictionary with position_sampling settings

    Returns:
        list: Tuples of (offset_x, offset_y, position_name)
              Offsets are in meters from tile center
    """
    pos_config = config['position_sampling']

    # If position sampling disabled, use center only
    if not pos_config['enabled']:
        return [(0.0, 0.0, 'center')]

    strategy = pos_config['strategy']
    positions = []

    # Calculate safe offset range
    # Sites must stay within tile bounds with minimum edge clearance
    tile_size_m = 1000.0
    min_edge_distance = pos_config['min_distance_to_edge_m']
    max_offset = (tile_size_m / 2.0) - site_radius_m - min_edge_distance

    # If site is too large for position variation, use center only
    if max_offset < 0:
        return [(0.0, 0.0, 'center')]

    # Add center position if requested
    if pos_config['include_center'] and strategy in ['center_only', 'systematic', 'mixed']:
        positions.append((0.0, 0.0, 'center'))

    # Generate random positions
    if strategy in ['random', 'mixed']:
        num_random = pos_config['num_random_positions']
        for i in range(num_random):
            # Random angle (0-360 degrees)
            angle = np.random.uniform(0, 2 * np.pi)
            # Random distance from center (0 to max_offset)
            distance = np.random.uniform(0, max_offset)

            # Convert polar to Cartesian coordinates
            offset_x = distance * np.cos(angle)
            offset_y = distance * np.sin(angle)

            positions.append((offset_x, offset_y, f'rand{i+1}'))

    # Ensure at least one position
    if not positions:
        positions.append((0.0, 0.0, 'center'))

    return positions


# ----------------------------------------------------------------------------
# Rotation and Cropping Functions
# ----------------------------------------------------------------------------

def rotate_and_crop_channels(channels_dict, rotation_angle, final_size):
    """
    Rotate all channels by the same angle and crop to final size.

    Used for positive samples where we extract a larger tile, rotate it,
    then crop the center to get the final grid with the site properly oriented.

    Args:
        channels_dict: Dictionary mapping channel names to 2D arrays
        rotation_angle: Rotation angle in degrees (0-360)
        final_size: Size of the cropped output (pixels)

    Returns:
        dict: Rotated and cropped channels (same keys as input)
    """
    rotated_channels = {}

    for channel_name, data in channels_dict.items():
        # Rotate array (reshape=False keeps original dimensions)
        if rotation_angle != 0:
            rotated = rotate(data, rotation_angle, reshape=False, order=1, mode='constant', cval=0)
        else:
            rotated = data

        # Crop center region
        h, w = rotated.shape
        center_h, center_w = h // 2, w // 2
        half_size = final_size // 2

        cropped = rotated[
            center_h - half_size : center_h + half_size,
            center_w - half_size : center_w + half_size
        ]

        rotated_channels[channel_name] = cropped.astype(np.float32)

    return rotated_channels


def rotate_entire_landscape(channels_dict, rotation_angle):
    """
    Rotate entire landscape tile without cropping.

    Used for integrated negatives where we rotate the whole landscape
    before extracting corner samples.

    Args:
        channels_dict: Dictionary mapping channel names to 2D arrays
        rotation_angle: Rotation angle in degrees (0-360)

    Returns:
        dict: Rotated channels at full size (same keys as input)
    """
    rotated_channels = {}

    for channel_name, data in channels_dict.items():
        if rotation_angle != 0:
            rotated = rotate(data, rotation_angle, reshape=False, order=1, mode='constant', cval=0)
        else:
            rotated = data

        rotated_channels[channel_name] = rotated.astype(np.float32)

    return rotated_channels


# ----------------------------------------------------------------------------
# Integrated Negative Sampling Functions
# ----------------------------------------------------------------------------

def extract_4_corners_from_rotated(rotated_channels, sampling_distance_px, slice_size_px):
    """
    Extract 4 corner slices from a rotated landscape.

    Used for integrated negatives: samples are taken from corners of the
    landscape tile, away from the central site location. This provides
    negative samples from the same geographic/environmental context.

    Args:
        rotated_channels: Dictionary of rotated channel arrays
        sampling_distance_px: Distance from center to corner sample centers (pixels)
        slice_size_px: Size of each corner slice (pixels)

    Returns:
        list: Four dictionaries (NW, NE, SW, SE), each containing channel slices

    Raises:
        ValueError: If corner slices would extend outside the landscape bounds
    """
    # Get dimensions from any channel (all should be same size)
    sample_channel = list(rotated_channels.values())[0]
    h, w = sample_channel.shape
    center_y, center_w = h // 2, w // 2

    half_slice = slice_size_px // 2

    # Define 4 corner positions (NW, NE, SW, SE)
    corner_positions = [
        ('NW', center_y - sampling_distance_px, center_w - sampling_distance_px),
        ('NE', center_y - sampling_distance_px, center_w + sampling_distance_px),
        ('SW', center_y + sampling_distance_px, center_w - sampling_distance_px),
        ('SE', center_y + sampling_distance_px, center_w + sampling_distance_px)
    ]

    corners = []
    for corner_name, cy, cx in corner_positions:
        corner_slice = {}

        # Calculate slice bounds
        y_start = int(cy - half_slice)
        y_end = int(cy + half_slice)
        x_start = int(cx - half_slice)
        x_end = int(cx + half_slice)

        # Validate bounds
        if y_start < 0 or y_end > h or x_start < 0 or x_end > w:
            raise ValueError(f"Corner {corner_name} slice out of bounds")

        # Extract slice from each channel
        for channel_name, data in rotated_channels.items():
            corner_slice[channel_name] = data[y_start:y_end, x_start:x_end].astype(np.float32)

        corners.append(corner_slice)

    return corners


def concatenate_corners(corners):
    """
    Concatenate 4 corner slices into a single 2x2 grid.

    Arranges corners as:
    +----+----+
    | NW | NE |
    +----+----+
    | SW | SE |
    +----+----+

    This creates a 100x100 pixel grid from four 50x50 corner samples.

    Args:
        corners: List of 4 dictionaries (NW, NE, SW, SE) with channel arrays

    Returns:
        dict: Combined channels in 2x2 grid layout
    """
    nw, ne, sw, se = corners

    concatenated = {}
    channel_names = list(nw.keys())

    for channel in channel_names:
        # Stack horizontally: top row (NW | NE), bottom row (SW | SE)
        top_row = np.hstack([nw[channel], ne[channel]])
        bottom_row = np.hstack([sw[channel], se[channel]])

        # Stack vertically: top + bottom
        combined = np.vstack([top_row, bottom_row])

        concatenated[channel] = combined.astype(np.float32)

    return concatenated


print("✓ Utility functions defined")

## 4. Initialize Zarr Arrays

In [ ]:
# ----------------------------------------------------------------------------
# 4.1 Calculate Dataset Dimensions
# ----------------------------------------------------------------------------

# Load site data to determine total dataset size
sites_df = pd.read_csv(config['paths']['input_sites'])
num_sites = len(sites_df)

# Determine rotation parameters
rotation_config = config['augmentation']['rotation_generation']
rotation_enabled = rotation_config['enabled']
rotation_step = rotation_config['rotation_step']

if rotation_enabled:
    num_rotations = int(360 / rotation_step)
    rotation_angles = [i * rotation_step for i in range(num_rotations)]
else:
    num_rotations = 1
    rotation_angles = [0]

# Determine position sampling parameters
pos_config = config['position_sampling']
if pos_config['enabled']:
    num_positions = (1 if pos_config['include_center'] else 0) + pos_config['num_random_positions']
else:
    num_positions = 1

# Calculate total grids per site and overall totals
grids_per_site = num_positions * num_rotations
total_positives = num_sites * grids_per_site

# Integrated negatives: one set of 4 corners per rotation per site
inegs_per_site = num_rotations
total_inegs = num_sites * inegs_per_site

# Landcover negatives: based on configured ratio
landcover_ratio = config['dataset_ratios']['landcover_to_positive_ratio']
total_landcover = int(total_positives * landcover_ratio)

# Configure batch sizes for chunked processing
# Batches allow processing large datasets without memory issues
sites_per_batch = min(25, num_sites)
num_site_batches = int(np.ceil(num_sites / sites_per_batch))
grids_per_pos_chunk = sites_per_batch * grids_per_site
grids_per_ineg_chunk = sites_per_batch * inegs_per_site

landcover_per_batch = min(200, max(50, total_landcover))
num_landcover_batches = int(np.ceil(total_landcover / landcover_per_batch))

# Display dimension summary
print("=" * 70)
print("DATASET DIMENSIONS")
print("=" * 70)
print(f"Sites:                     {num_sites}")
print(f"Positions per site:        {num_positions}")
print(f"Rotations:                 {num_rotations} (angles: {rotation_angles})")
print(f"Grids per site:            {grids_per_site}")
print()
print("Total grids:")
print(f"  Positives:               {total_positives:,}")
print(f"  Integrated negatives:    {total_inegs:,}")
print(f"  Landcover negatives:     {total_landcover:,}")
print(f"  TOTAL:                   {total_positives + total_inegs + total_landcover:,}")
print()
print("Batch configuration:")
print(f"  Site batches:            {num_site_batches}")
print(f"  Grids per pos chunk:     {grids_per_pos_chunk}")
print(f"  Grids per ineg chunk:    {grids_per_ineg_chunk}")
print(f"  Landcover batches:       {num_landcover_batches}")
print(f"  Grids per lc chunk:      {landcover_per_batch}")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# 4.2 Create Zarr Arrays on GCS
# ----------------------------------------------------------------------------

# Initialize GCS filesystem
fs = gcsfs.GCSFileSystem()

# Compression settings for Zarr v3
# zstd at level 5 provides good compression with reasonable speed
compressor = BloscCodec(cname='zstd', clevel=5)

print("\n" + "=" * 70)
print("CREATING ZARR ARRAYS ON GCS")
print("=" * 70)

# --- Positives ---
# Contains archaeological sites with multiple label variants
pos_store = fs.get_mapper(f'{DATASET_PATH}positives.zarr')
pos_root = zarr.group(store=pos_store, overwrite=True)

# Grid data: (samples, channels, height, width)
pos_grids = pos_root.create_array(
    'grids',
    shape=(total_positives, 11, 100, 100),
    chunks=(grids_per_pos_chunk, 11, 100, 100),
    dtype='float32',
    compressor=compressor,
    fill_value=0
)

# Hard labels: binary masks
pos_labels_hard = pos_root.create_array(
    'labels_hard',
    shape=(total_positives, 100, 100),
    chunks=(grids_per_pos_chunk, 100, 100),
    dtype='uint8',
    compressor=compressor,
    fill_value=0
)

# Soft labels: probabilistic masks with different sigma values
# Small sigma (1): tight focus on boundaries
# Medium sigma (3): moderate spatial extent
# Large sigma (8): broad context
for sigma in [1, 3, 8]:
    pos_root.create_array(
        f'labels_soft_sigma{sigma}',
        shape=(total_positives, 100, 100),
        chunks=(grids_per_pos_chunk, 100, 100),
        dtype='float16',  # float16 sufficient for probabilities, saves space
        compressor=compressor,
        fill_value=0
    )

print(f"✓ Positives: {total_positives:,} grids in {num_site_batches} chunks")

# --- Integrated Negatives ---
# Corner samples from same landscapes as positives, but away from sites
ineg_store = fs.get_mapper(f'{DATASET_PATH}integrated_negatives.zarr')
ineg_root = zarr.group(store=ineg_store, overwrite=True)

ineg_grids = ineg_root.create_array(
    'grids',
    shape=(total_inegs, 11, 100, 100),
    chunks=(grids_per_ineg_chunk, 11, 100, 100),
    dtype='float32',
    compressor=compressor,
    fill_value=0
)

ineg_labels = ineg_root.create_array(
    'labels',
    shape=(total_inegs, 100, 100),
    chunks=(grids_per_ineg_chunk, 100, 100),
    dtype='uint8',
    compressor=compressor,
    fill_value=0
)

print(f"✓ Integrated negatives: {total_inegs:,} grids in {num_site_batches} chunks")

# --- Landcover Negatives ---
# Samples from different landcover types (urban, water, cropland)
lneg_store = fs.get_mapper(f'{DATASET_PATH}landcover_negatives.zarr')
lneg_root = zarr.group(store=lneg_store, overwrite=True)

lneg_grids = lneg_root.create_array(
    'grids',
    shape=(total_landcover, 11, 100, 100),
    chunks=(landcover_per_batch, 11, 100, 100),
    dtype='float32',
    compressor=compressor,
    fill_value=0
)

lneg_labels = lneg_root.create_array(
    'labels',
    shape=(total_landcover, 100, 100),
    chunks=(landcover_per_batch, 100, 100),
    dtype='uint8',
    compressor=compressor,
    fill_value=0
)

print(f"✓ Landcover negatives: {total_landcover:,} grids in {num_landcover_batches} chunks")
print(f"\n✓ All Zarr arrays initialized at {DATASET_PATH}")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# 4.3 Initialize Progress Tracker
# ----------------------------------------------------------------------------

# Progress tracker enables resuming interrupted runs
progress_path = f'{DATASET_PATH}progress.json'
tracker = ProgressTracker(progress_path)

summary = tracker.get_summary()
print("\n" + "=" * 70)
print("PROGRESS TRACKER INITIALIZED")
print("=" * 70)
print(f"Site batches completed:      {summary['site_batches_completed']}/{num_site_batches}")
print(f"Landcover batches completed: {summary['landcover_batches_completed']}/{num_landcover_batches}")
print(f"Last updated:                {summary['last_updated']}")
print("=" * 70)

## 5. Process Site Batches (Positives & Integrated Negatives)

In [ ]:
# ----------------------------------------------------------------------------
# 5.1 Initialize Processors
# ----------------------------------------------------------------------------

# Calculate extraction size with rotation buffer
# We extract a larger tile than needed, rotate it, then crop the center
# This prevents empty corners that would appear if we rotated the exact size
rotation_buffer = config.get('augmentation', {}).get('rotation_buffer_factor', 1.0)
base_cell_size = config['grid']['cell_size_km']
base_pixels_per_km = config['grid']['pixels_per_km']

extraction_cell_size = base_cell_size * rotation_buffer
extraction_pixels_per_side = int(base_pixels_per_km * extraction_cell_size)

# Temporarily modify config for extraction
# We'll restore original values after processing
original_cell_size = config['grid']['cell_size_km']
original_pixels_per_km = config['grid']['pixels_per_km']

config['grid']['cell_size_km'] = extraction_cell_size
config['grid']['pixels_per_km'] = extraction_pixels_per_side

# Initialize processors
extractor = GEEExtractor(config)
label_generator = LabelGenerator(config, config['paths']['contours_geojson'])

# Initialize metadata collectors
# These will store information about each generated grid
pos_metadata = []
ineg_metadata = []

print("=" * 70)
print("PROCESSORS INITIALIZED")
print("=" * 70)
print(f"Extraction size:  {extraction_cell_size} km ({extraction_pixels_per_side}×{extraction_pixels_per_side} px)")
print(f"Final size:       {base_cell_size} km ({base_pixels_per_km}×{base_pixels_per_km} px)")
print(f"Rotation buffer:  {rotation_buffer}x")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# 5.2 Process Each Site Batch
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("PROCESSING SITE BATCHES")
print("=" * 70)

# Buffer margin for integrated negative sampling
buffer_margin = config['integrated_negatives']['buffer_margin']

for batch_id in tqdm(range(num_site_batches), desc="Site batches"):
    # Skip if already completed (enables resuming)
    if tracker.is_site_batch_complete(batch_id):
        print(f"\nBatch {batch_id}: Already completed, skipping")
        continue

    print(f"\n{'=' * 70}")
    print(f"BATCH {batch_id}/{num_site_batches-1}")
    print(f"{'=' * 70}")

    # Get sites for this batch
    start_idx = batch_id * sites_per_batch
    end_idx = min(start_idx + sites_per_batch, num_sites)
    batch_sites = sites_df.iloc[start_idx:end_idx]

    print(f"Processing sites {start_idx} to {end_idx-1} ({len(batch_sites)} sites)")

    # Initialize buffers for this batch
    pos_grids_buffer = []
    pos_labels_buffer = {
        'hard': [],
        'sigma1': [],
        'sigma3': [],
        'sigma8': []
    }
    ineg_grids_buffer = []
    ineg_labels_buffer = []

    batch_pos_meta = []
    batch_ineg_meta = []

    # Process each site in the batch
    for site_idx, (_, row) in enumerate(batch_sites.iterrows()):
        site_id = row['site_id']
        site_lat = row['latitude']
        site_lon = row['longitude']
        site_type = row.get('site_type', 'unknown')

        global_site_idx = start_idx + site_idx

        try:
            # --- POSITIVE SAMPLES ---

            # Calculate position offsets to avoid center bias
            site_radius = calculate_site_radius(row)
            positions = generate_position_offsets(site_radius, config)

            # Extract large tile for rotation and cropping
            channels_pos, gee_metadata = extractor.extract_all_channels(site_lat, site_lon)
            indices_pos = IndexCalculator.calculate_all_indices(channels_pos)
            all_channels_pos = {**channels_pos, **indices_pos}

            # Generate positive samples for each position × rotation combination
            for offset_x_m, offset_y_m, position_name in positions:
                # Convert meter offsets to lat/lon offsets
                meters_per_deg_lat = 111320.0
                meters_per_deg_lon = 111320.0 * np.cos(np.radians(site_lat))

                extraction_lat = site_lat + (offset_y_m / meters_per_deg_lat)
                extraction_lon = site_lon + (offset_x_m / meters_per_deg_lon)

                for rotation_angle in rotation_angles:
                    # Rotate and crop to final size
                    grid = rotate_and_crop_channels(
                        all_channels_pos,
                        rotation_angle,
                        base_pixels_per_km
                    )

                    # Stack channels in standard order
                    grid_array = np.stack([grid[ch] for ch in CHANNEL_ORDER], axis=0)
                    pos_grids_buffer.append(grid_array)

                    # Generate all label variants (hard + soft with multiple sigmas)
                    labels = label_generator.generate_positive_labels(
                        extraction_lat,
                        extraction_lon,
                        rotation_angle
                    )

                    pos_labels_buffer['hard'].append(labels['hard'])
                    pos_labels_buffer['sigma1'].append(labels['sigma1'])
                    pos_labels_buffer['sigma3'].append(labels['sigma3'])
                    pos_labels_buffer['sigma8'].append(labels['sigma8'])

                    # Record metadata
                    zarr_idx = batch_id * grids_per_pos_chunk + len(pos_grids_buffer) - 1
                    grid_id = f"grid_{global_site_idx+1:06d}_pos_{position_name}_rot{int(rotation_angle):03d}"

                    batch_pos_meta.append({
                        'zarr_index': zarr_idx,
                        'grid_id': grid_id,
                        'site_id': site_id,
                        'site_batch': batch_id,
                        'chunk_index': batch_id,
                        'position': position_name,
                        'rotation': rotation_angle,
                        'lat': extraction_lat,
                        'lon': extraction_lon,
                        'label': 1,
                        'site_type': site_type
                    })

            # Clean up positive sample data
            del channels_pos, indices_pos, all_channels_pos
            gc.collect()

            # --- INTEGRATED NEGATIVES ---

            # Extract larger tile to accommodate corner sampling away from site
            sampling_distance_m = site_radius + 250 + buffer_margin
            required_radius_m = sampling_distance_m + 250  # +250 for corner slice size
            extraction_size_km_ineg = (required_radius_m * 2 / 1000.0) * rotation_buffer
            extraction_pixels_ineg = int(extraction_size_km_ineg * 100)

            # Temporarily update config for this larger extraction
            config['grid']['cell_size_km'] = extraction_size_km_ineg
            config['grid']['pixels_per_km'] = extraction_pixels_ineg

            # Extract channels for integrated negatives
            channels_ineg, _ = extractor.extract_all_channels(site_lat, site_lon)
            indices_ineg = IndexCalculator.calculate_all_indices(channels_ineg)
            all_channels_ineg = {**channels_ineg, **indices_ineg}

            # Calculate sampling distance in pixels
            pixels_per_meter = extraction_pixels_ineg / (extraction_size_km_ineg * 1000)
            sampling_distance_px = int(sampling_distance_m * pixels_per_meter)

            # Generate integrated negatives for each rotation
            for rotation_angle in rotation_angles:
                # Rotate entire landscape
                rotated_landscape = rotate_entire_landscape(all_channels_ineg, rotation_angle)

                # Extract 4 corners from rotated landscape
                corners = extract_4_corners_from_rotated(rotated_landscape, sampling_distance_px, 50)

                # Concatenate corners into 2x2 grid (100x100 total)
                concatenated = concatenate_corners(corners)

                # Stack channels in standard order
                grid_array = np.stack([concatenated[ch] for ch in CHANNEL_ORDER], axis=0)
                ineg_grids_buffer.append(grid_array)

                # Generate zero labels for negatives
                zero_label = label_generator.generate_negative_labels()
                ineg_labels_buffer.append(zero_label)

                # Record metadata
                zarr_idx = batch_id * grids_per_ineg_chunk + len(ineg_grids_buffer) - 1
                grid_id = f"ineg_{global_site_idx+1:06d}_rot{int(rotation_angle):03d}"

                batch_ineg_meta.append({
                    'zarr_index': zarr_idx,
                    'grid_id': grid_id,
                    'source_site_id': site_id,
                    'site_batch': batch_id,
                    'chunk_index': batch_id,
                    'rotation': rotation_angle,
                    'lat': site_lat,
                    'lon': site_lon,
                    'label': 0,
                    'neg_type': 'integrated_context'
                })

            # Restore config to standard extraction size
            config['grid']['cell_size_km'] = extraction_cell_size
            config['grid']['pixels_per_km'] = extraction_pixels_per_side

            # Clean up integrated negative data
            del channels_ineg, indices_ineg, all_channels_ineg
            gc.collect()

        except Exception as e:
            print(f"  ✗ Site {global_site_idx} ({site_id}) failed: {str(e)}")
            continue

    # --- WRITE BATCH TO ZARR ---

    print(f"\nWriting batch {batch_id} to Zarr...")

    # Write positives
    start_pos = batch_id * grids_per_pos_chunk
    end_pos = start_pos + len(pos_grids_buffer)

    pos_grids[start_pos:end_pos] = np.stack(pos_grids_buffer)
    pos_labels_hard[start_pos:end_pos] = np.stack(pos_labels_buffer['hard'])
    pos_root['labels_soft_sigma1'][start_pos:end_pos] = np.stack(pos_labels_buffer['sigma1'])
    pos_root['labels_soft_sigma3'][start_pos:end_pos] = np.stack(pos_labels_buffer['sigma3'])
    pos_root['labels_soft_sigma8'][start_pos:end_pos] = np.stack(pos_labels_buffer['sigma8'])

    # Write integrated negatives
    start_ineg = batch_id * grids_per_ineg_chunk
    end_ineg = start_ineg + len(ineg_grids_buffer)

    ineg_grids[start_ineg:end_ineg] = np.stack(ineg_grids_buffer)
    ineg_labels[start_ineg:end_ineg] = np.stack(ineg_labels_buffer)

    print(f"  ✓ Wrote {len(pos_grids_buffer)} positives, {len(ineg_grids_buffer)} integrated negatives")

    # Update metadata collectors
    pos_metadata.extend(batch_pos_meta)
    ineg_metadata.extend(batch_ineg_meta)

    # Mark batch as complete (enables resuming)
    tracker.mark_site_batch_complete(batch_id)

    # Clear buffers to free memory
    del pos_grids_buffer, pos_labels_buffer, ineg_grids_buffer, ineg_labels_buffer
    del batch_pos_meta, batch_ineg_meta
    gc.collect()

    # Rate limiting to avoid overwhelming GEE API
    time.sleep(2)

# Restore original config values
config['grid']['cell_size_km'] = original_cell_size
config['grid']['pixels_per_km'] = original_pixels_per_km

print("\n" + "=" * 70)
print("SITE BATCHES COMPLETE")
print("=" * 70)
print(f"Positives:              {len(pos_metadata):,}")
print(f"Integrated negatives:   {len(ineg_metadata):,}")
print("=" * 70)

## 6. Process Site Batches (Landcover)

In [ ]:
# ----------------------------------------------------------------------------
# 6.1 Landcover Coordinate Generation Functions
# ----------------------------------------------------------------------------

def get_base_coordinates():
    """
    Define base coordinates for different landcover types.

    These are manually selected locations representing typical examples
    of each landcover type in the study region (Peru and Bolivia):
    - Urban: Cities and towns
    - Water: Lakes, rivers, reservoirs, ocean
    - Cropland: Agricultural valleys and plains

    Returns:
        dict: Maps landcover type to list of (lat, lon, description) tuples
    """
    urban_coords = [
        (-12.0464, -77.0428, "Lima, Peru"),
        (-16.4090, -71.5375, "Arequipa, Peru"),
        (-13.5319, -71.9675, "Cusco, Peru"),
        (-11.0187, -76.8686, "Huancayo, Peru"),
        (-8.3791, -74.5539, "Pucallpa, Peru"),
        (-12.5897, -69.1890, "Puerto Maldonado, Peru"),
        (-16.5000, -68.1500, "La Paz, Bolivia"),
        (-17.7833, -63.1821, "Santa Cruz, Bolivia"),
        (-19.0333, -65.2627, "Sucre, Bolivia"),
        (-17.3895, -66.1568, "Cochabamba, Bolivia"),
        (-11.0049, -77.6050, "Huacho, Peru"),
        (-14.0650, -75.7350, "Ica, Peru"),
        (-8.1116, -79.0288, "Trujillo, Peru"),
        (-6.7014, -79.9061, "Chiclayo, Peru"),
        (-5.1973, -80.6326, "Piura, Peru"),
    ]

    water_coords = [
        (-15.8422, -69.9406, "Lake Titicaca (south)"),
        (-15.7333, -69.8667, "Lake Titicaca (central)"),
        (-16.0000, -69.0500, "Lake Titicaca (east)"),
        (-15.5500, -70.0500, "Lake Titicaca (north)"),
        (-12.0000, -77.1000, "Pacific Ocean - Lima coast"),
        (-16.4000, -72.5000, "Pacific Ocean - Arequipa coast"),
        (-8.0000, -79.0000, "Pacific Ocean - Trujillo coast"),
        (-13.2000, -76.2000, "Pacific Ocean - Pisco coast"),
        (-11.8000, -75.2000, "Rio Mantaro area"),
        (-12.5000, -69.0000, "Rio Madre de Dios"),
        (-8.5000, -74.0000, "Rio Ucayali"),
        (-4.5000, -73.2000, "Amazon River - Peru"),
        (-17.9667, -67.1167, "Lake Poopó, Bolivia"),
        (-14.0000, -76.0000, "Reserva Nacional de Paracas"),
        (-13.7000, -73.7000, "Rio Apurímac"),
    ]

    cropland_coords = [
        (-12.2000, -76.9000, "Cañete Valley - Lima"),
        (-13.4000, -76.1000, "Pisco Valley"),
        (-14.8000, -75.9000, "Palpa Valley"),
        (-11.2000, -75.9000, "Junín agricultural area"),
        (-8.6000, -78.5000, "Chicama Valley - Trujillo"),
        (-6.9000, -79.7000, "Lambayeque Valley"),
        (-5.5000, -80.4000, "Piura agricultural region"),
        (-16.3000, -71.6000, "Arequipa agricultural valley"),
        (-17.8000, -63.0000, "Santa Cruz agricultural plain, Bolivia"),
        (-12.7000, -76.2000, "Mala Valley"),
        (-13.7000, -76.5000, "Chincha Valley"),
        (-15.5000, -71.5000, "Majes Valley"),
        (-17.0000, -66.0000, "Cochabamba Valley, Bolivia"),
        (-11.5000, -77.5000, "Huaral Valley"),
        (-9.5000, -77.5000, "Casma Valley"),
    ]

    return {
        'urban': urban_coords,
        'water': water_coords,
        'cropland': cropland_coords
    }


def generate_variations(base_coords, count, max_radius_km=5.0, min_spacing_km=1.5):
    """
    Generate spatial variations around base coordinates.

    Creates multiple samples per base location by adding random offsets.
    This provides diversity while maintaining the landcover type.

    Args:
        base_coords: List of (lat, lon, description) tuples
        count: Total number of variations to generate
        max_radius_km: Maximum distance from base point (km)
        min_spacing_km: Minimum distance between generated points (km)

    Returns:
        list: Generated (lat, lon, description) tuples with spatial offsets
    """
    variations = []
    samples_per_base = int(np.ceil(count / len(base_coords)))

    for base_lat, base_lon, base_name in base_coords:
        if len(variations) >= count:
            break

        base_variations = []
        attempts = 0
        max_attempts = samples_per_base * 10

        while len(base_variations) < samples_per_base and attempts < max_attempts:
            attempts += 1

            # Calculate degrees per km at this latitude
            lat_deg_per_km = 1.0 / 111.32
            lon_deg_per_km = 1.0 / (111.32 * np.cos(np.radians(base_lat)))

            # Generate random offset in polar coordinates
            distance_km = np.random.uniform(0, max_radius_km)
            angle = np.random.uniform(0, 2 * np.pi)

            # Convert to lat/lon offsets
            lat_offset = distance_km * np.cos(angle) * lat_deg_per_km
            lon_offset = distance_km * np.sin(angle) * lon_deg_per_km

            new_lat = base_lat + lat_offset
            new_lon = base_lon + lon_offset

            # Check minimum spacing constraint
            too_close = False
            min_spacing_deg = min_spacing_km * lat_deg_per_km

            for existing_lat, existing_lon, _ in base_variations:
                dist = np.sqrt((new_lat - existing_lat)**2 + (new_lon - existing_lon)**2)
                if dist < min_spacing_deg:
                    too_close = True
                    break

            if not too_close:
                description = f"{base_name} +{distance_km:.1f}km"
                base_variations.append((new_lat, new_lon, description))

        variations.extend(base_variations[:samples_per_base])

    return variations[:count]


print("✓ Landcover functions defined")

In [ ]:
# ----------------------------------------------------------------------------
# 6.2 Process Each Landcover Batch
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("PROCESSING LANDCOVER BATCHES")
print("=" * 70)

# Get landcover ratios from config
landcover_config = config['label_0_composition']['landcover']
urban_ratio = landcover_config['urban_ratio']
water_ratio = landcover_config['water_ratio']
cropland_ratio = landcover_config['cropland_ratio']

# Get base coordinates for each landcover type
base_coords = get_base_coordinates()
lneg_metadata = []

for batch_id in tqdm(range(num_landcover_batches), desc="Landcover batches"):
    # Skip if already completed (enables resuming)
    if tracker.is_landcover_batch_complete(batch_id):
        print(f"\nBatch {batch_id}: Already completed, skipping")
        continue

    print(f"\n{'=' * 70}")
    print(f"LANDCOVER BATCH {batch_id}/{num_landcover_batches-1}")
    print(f"{'=' * 70}")

    # Calculate distribution of landcover types for this batch
    grids_this_batch = min(landcover_per_batch, total_landcover - batch_id * landcover_per_batch)

    urban_count = int(grids_this_batch * urban_ratio)
    water_count = int(grids_this_batch * water_ratio)
    cropland_count = grids_this_batch - urban_count - water_count

    print(f"Generating {grids_this_batch} samples:")
    print(f"  Urban:     {urban_count}")
    print(f"  Water:     {water_count}")
    print(f"  Cropland:  {cropland_count}")

    # Generate coordinate variations for each landcover type
    all_samples = {}
    all_samples['urban'] = generate_variations(base_coords['urban'], urban_count, max_radius_km=8.0)
    all_samples['water'] = generate_variations(base_coords['water'], water_count, max_radius_km=10.0)
    all_samples['cropland'] = generate_variations(base_coords['cropland'], cropland_count, max_radius_km=6.0)

    # Combine all samples into single list
    all_coords = []
    for category, samples in all_samples.items():
        for lat, lon, desc in samples:
            all_coords.append((lat, lon, category, desc))

    # Initialize buffers for this batch
    lneg_grids_buffer = []
    lneg_labels_buffer = []
    batch_lneg_meta = []

    # Process each landcover sample
    for idx, (lat, lon, category, desc) in enumerate(all_coords):
        try:
            # Extract satellite imagery and elevation data
            channels, gee_metadata = extractor.extract_all_channels(lat, lon)
            indices = IndexCalculator.calculate_all_indices(channels)

            # Stack channels in standard order
            grid_array = np.stack([channels[ch] if ch in channels else indices[ch] for ch in CHANNEL_ORDER], axis=0)
            lneg_grids_buffer.append(grid_array)

            # Generate zero labels (no archaeological features)
            zero_label = label_generator.generate_negative_labels()
            lneg_labels_buffer.append(zero_label)

            # Record metadata
            zarr_idx = batch_id * landcover_per_batch + len(lneg_grids_buffer) - 1
            grid_id = f"lneg_{zarr_idx:06d}"

            batch_lneg_meta.append({
                'zarr_index': zarr_idx,
                'grid_id': grid_id,
                'landcover_batch': batch_id,
                'chunk_index': batch_id,
                'lat': lat,
                'lon': lon,
                'label': 0,
                'neg_type': category,
                'location_description': desc
            })

            # Clean up
            del channels, indices
            gc.collect()

        except Exception as e:
            print(f"  ✗ Failed {category} at ({lat:.4f}, {lon:.4f}): {str(e)}")
            continue

    # --- WRITE BATCH TO ZARR ---

    print(f"\nWriting batch {batch_id} to Zarr...")

    start_idx = batch_id * landcover_per_batch
    end_idx = start_idx + len(lneg_grids_buffer)

    lneg_grids[start_idx:end_idx] = np.stack(lneg_grids_buffer)
    lneg_labels[start_idx:end_idx] = np.stack(lneg_labels_buffer)

    print(f"  ✓ Wrote {len(lneg_grids_buffer)} landcover negatives")

    # Update metadata collector
    lneg_metadata.extend(batch_lneg_meta)

    # Mark batch as complete (enables resuming)
    tracker.mark_landcover_batch_complete(batch_id)

    # Clear buffers to free memory
    del lneg_grids_buffer, lneg_labels_buffer, batch_lneg_meta
    gc.collect()

    # Rate limiting to avoid overwhelming GEE API
    time.sleep(2)

print("\n" + "=" * 70)
print("LANDCOVER BATCHES COMPLETE")
print("=" * 70)
print(f"Landcover negatives:   {len(lneg_metadata):,}")
print("=" * 70)

## 7. Save Metadata & Validation

In [ ]:
# ----------------------------------------------------------------------------
# 7.1 Save Metadata to GCS
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("SAVING METADATA")
print("=" * 70)

# Convert metadata lists to DataFrames
pos_df = pd.DataFrame(pos_metadata)
ineg_df = pd.DataFrame(ineg_metadata)
lneg_df = pd.DataFrame(lneg_metadata)

# Save to GCS as Parquet (efficient columnar format)
metadata_path = f'{DATASET_PATH}metadata/'

with fs.open(f'{metadata_path}positives.parquet', 'wb') as f:
    pos_df.to_parquet(f, index=False)

with fs.open(f'{metadata_path}integrated_negatives.parquet', 'wb') as f:
    ineg_df.to_parquet(f, index=False)

with fs.open(f'{metadata_path}landcover_negatives.parquet', 'wb') as f:
    lneg_df.to_parquet(f, index=False)

print(f"✓ Metadata saved to {metadata_path}")
print(f"  Positives:              {len(pos_df):,} records")
print(f"  Integrated negatives:   {len(ineg_df):,} records")
print(f"  Landcover negatives:    {len(lneg_df):,} records")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# 7.2 Validation Checks
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

# --- Check Zarr Array Shapes ---
print("\nZarr array shapes:")
print(f"  Positives/grids:                {pos_grids.shape}")
print(f"  Positives/labels_hard:          {pos_labels_hard.shape}")
print(f"  Positives/labels_soft_sigma1:   {pos_root['labels_soft_sigma1'].shape}")
print(f"  Positives/labels_soft_sigma3:   {pos_root['labels_soft_sigma3'].shape}")
print(f"  Positives/labels_soft_sigma8:   {pos_root['labels_soft_sigma8'].shape}")
print(f"  Integrated negatives/grids:     {ineg_grids.shape}")
print(f"  Integrated negatives/labels:    {ineg_labels.shape}")
print(f"  Landcover negatives/grids:      {lneg_grids.shape}")
print(f"  Landcover negatives/labels:     {lneg_labels.shape}")

# --- Check for NaN/Inf Values ---
print("\nData quality checks:")

# Check first positive sample
sample_pos = pos_grids[0]
has_nan_pos = np.isnan(sample_pos).any()
has_inf_pos = np.isinf(sample_pos).any()
print(f"  Positives - NaN: {has_nan_pos}, Inf: {has_inf_pos}")

# Check first integrated negative sample
sample_ineg = ineg_grids[0]
has_nan_ineg = np.isnan(sample_ineg).any()
has_inf_ineg = np.isinf(sample_ineg).any()
print(f"  Integrated negatives - NaN: {has_nan_ineg}, Inf: {has_inf_ineg}")

# Check first landcover negative sample
sample_lneg = lneg_grids[0]
has_nan_lneg = np.isnan(sample_lneg).any()
has_inf_lneg = np.isinf(sample_lneg).any()
print(f"  Landcover negatives - NaN: {has_nan_lneg}, Inf: {has_inf_lneg}")

# --- Sample Label Statistics ---
print("\nSample label statistics:")

# Hard label
sample_hard = pos_labels_hard[0]
unique_vals = np.unique(sample_hard)
print(f"  Hard label - unique values: {unique_vals}")
print(f"  Hard label - pixel count: {sample_hard.sum()}")

# Soft labels with different sigma values
sample_sigma1 = pos_root['labels_soft_sigma1'][0]
print(f"  Sigma1 - min: {sample_sigma1.min():.4f}, max: {sample_sigma1.max():.4f}, mean: {sample_sigma1.mean():.4f}")

sample_sigma3 = pos_root['labels_soft_sigma3'][0]
print(f"  Sigma3 - min: {sample_sigma3.min():.4f}, max: {sample_sigma3.max():.4f}, mean: {sample_sigma3.mean():.4f}")

sample_sigma8 = pos_root['labels_soft_sigma8'][0]
print(f"  Sigma8 - min: {sample_sigma8.min():.4f}, max: {sample_sigma8.max():.4f}, mean: {sample_sigma8.mean():.4f}")

# --- Dataset Composition Summary ---
print("\nDataset composition:")
print(f"  Total samples:              {len(pos_df) + len(ineg_df) + len(lneg_df):,}")
print(f"  Positive samples:           {len(pos_df):,} ({len(pos_df)/(len(pos_df)+len(ineg_df)+len(lneg_df))*100:.1f}%)")
print(f"  Integrated negatives:       {len(ineg_df):,} ({len(ineg_df)/(len(pos_df)+len(ineg_df)+len(lneg_df))*100:.1f}%)")
print(f"  Landcover negatives:        {len(lneg_df):,} ({len(lneg_df)/(len(pos_df)+len(ineg_df)+len(lneg_df))*100:.1f}%)")

# --- Landcover Type Distribution ---
if 'neg_type' in lneg_df.columns:
    print("\nLandcover distribution:")
    for neg_type in lneg_df['neg_type'].unique():
        count = (lneg_df['neg_type'] == neg_type).sum()
        print(f"  {neg_type.capitalize()}: {count:,} ({count/len(lneg_df)*100:.1f}%)")

print("\n✓ Validation complete")
print("=" * 70)

In [ ]:
# ----------------------------------------------------------------------------
# 7.3 Final Summary
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET GENERATION COMPLETE")
print("=" * 70)

summary = tracker.get_summary()

print(f"\nDataset location: {DATASET_PATH}")

print(f"\nFinal counts:")
print(f"  Positives:              {len(pos_df):,}")
print(f"  Integrated negatives:   {len(ineg_df):,}")
print(f"  Landcover negatives:    {len(lneg_df):,}")
print(f"  TOTAL:                  {len(pos_df) + len(ineg_df) + len(lneg_df):,}")

print(f"\nProgress:")
print(f"  Site batches:           {summary['site_batches_completed']}/{num_site_batches}")
print(f"  Landcover batches:      {summary['landcover_batches_completed']}/{num_landcover_batches}")
print(f"  Last updated:           {summary['last_updated']}")

print(f"\nStorage structure:")
print(f"  {DATASET_PATH}positives.zarr/")
print(f"    ├── grids/                    {pos_grids.shape}")
print(f"    ├── labels_hard/              {pos_labels_hard.shape}")
print(f"    ├── labels_soft_sigma1/       {pos_root['labels_soft_sigma1'].shape}")
print(f"    ├── labels_soft_sigma3/       {pos_root['labels_soft_sigma3'].shape}")
print(f"    └── labels_soft_sigma8/       {pos_root['labels_soft_sigma8'].shape}")
print(f"  {DATASET_PATH}integrated_negatives.zarr/")
print(f"    ├── grids/                    {ineg_grids.shape}")
print(f"    └── labels/                   {ineg_labels.shape}")
print(f"  {DATASET_PATH}landcover_negatives.zarr/")
print(f"    ├── grids/                    {lneg_grids.shape}")
print(f"    └── labels/                   {lneg_labels.shape}")
print(f"  {DATASET_PATH}metadata/")
print(f"    ├── positives.parquet")
print(f"    ├── integrated_negatives.parquet")
print(f"    └── landcover_negatives.parquet")

print("\n" + "=" * 70)
print("✓ All data successfully written to GCS")
print("=" * 70)

In [ ]:
# Visualization fucntions
def visualize_sample_with_labels(zarr_index=0, sample_type='positive'):
    """
    Visualize one sample with RGB and all its labels.

    Args:
        zarr_index: Index in the Zarr array
        sample_type: 'positive', 'integrated', or 'landcover'
    """
    if sample_type == 'positive':
        grid = pos_grids[zarr_index]
        meta = pos_df[pos_df['zarr_index'] == zarr_index].iloc[0]

        # Load all labels
        label_hard = pos_labels_hard[zarr_index]
        label_sigma1 = pos_root['labels_soft_sigma1'][zarr_index]
        label_sigma3 = pos_root['labels_soft_sigma3'][zarr_index]
        label_sigma8 = pos_root['labels_soft_sigma8'][zarr_index]

        title = f"Positive: {meta['grid_id']}"

    elif sample_type == 'integrated':
        grid = ineg_grids[zarr_index]
        meta = ineg_df[ineg_df['zarr_index'] == zarr_index].iloc[0]

        label_hard = ineg_labels[zarr_index]
        label_sigma1 = label_sigma3 = label_sigma8 = None

        title = f"Integrated Negative: {meta['grid_id']}"

    else:  # landcover
        grid = lneg_grids[zarr_index]
        meta = lneg_df[lneg_df['zarr_index'] == zarr_index].iloc[0]

        label_hard = lneg_labels[zarr_index]
        label_sigma1 = label_sigma3 = label_sigma8 = None

        title = f"Landcover Negative ({meta['neg_type']}): {meta['grid_id']}"

    # Extract RGB (B4=Red, B3=Green, B2=Blue)
    r = grid[2]  # B4
    g = grid[1]  # B3
    b = grid[0]  # B2

    rgb = np.stack([r, g, b], axis=-1)
    p2, p98 = np.percentile(rgb, [2, 98])
    rgb_norm = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    # Create plot
    if sample_type == 'positive':
        fig, axes = plt.subplots(1, 5, figsize=(25, 5))

        # RGB
        axes[0].imshow(rgb_norm)
        axes[0].set_title('RGB Composite', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Hard label
        axes[1].imshow(label_hard, cmap='gray', vmin=0, vmax=1)
        axes[1].set_title(f'Hard Label\n{label_hard.sum()} pixels', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # Sigma 1
        im2 = axes[2].imshow(label_sigma1, cmap='hot', vmin=0, vmax=1)
        axes[2].set_title(f'Soft Label (σ=1)\nMax: {label_sigma1.max():.3f}', fontsize=14, fontweight='bold')
        axes[2].axis('off')
        plt.colorbar(im2, ax=axes[2], fraction=0.046)

        # Sigma 3
        im3 = axes[3].imshow(label_sigma3, cmap='hot', vmin=0, vmax=1)
        axes[3].set_title(f'Soft Label (σ=3)\nMax: {label_sigma3.max():.3f}', fontsize=14, fontweight='bold')
        axes[3].axis('off')
        plt.colorbar(im3, ax=axes[3], fraction=0.046)

        # Sigma 8
        im4 = axes[4].imshow(label_sigma8, cmap='hot', vmin=0, vmax=1)
        axes[4].set_title(f'Soft Label (σ=8)\nMax: {label_sigma8.max():.3f}', fontsize=14, fontweight='bold')
        axes[4].axis('off')
        plt.colorbar(im4, ax=axes[4], fraction=0.046)

    else:  # Negatives
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # RGB
        axes[0].imshow(rgb_norm)
        axes[0].set_title('RGB Composite', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # Label (all zeros)
        axes[1].imshow(label_hard, cmap='gray', vmin=0, vmax=1)
        axes[1].set_title(f'Label (All Zeros)\nSum: {label_hard.sum()}', fontsize=14, fontweight='bold')
        axes[1].axis('off')

    fig.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


def visualize_all_variants_for_site(site_index=0):
    """
    Visualize ALL variants for one site: all positions × all rotations × all labels.
    Shows complete augmentation pipeline for one site.

    Args:
        site_index: Site index (0-based)
    """
    # Find all grids for this site
    pattern = f'grid_{site_index+1:06d}_'
    site_grids = pos_df[pos_df['grid_id'].str.contains(pattern)]

    if site_grids.shape[0] == 0:
        print(f"No grids found for site {site_index+1}")
        return

    # Get unique positions and rotations
    positions = sorted(site_grids['position'].unique())
    rotations = sorted(site_grids['rotation'].unique())

    print(f"\nSite {site_index+1:06d}")
    print(f"  Positions: {positions}")
    print(f"  Rotations: {rotations}")
    print(f"  Total variants: {len(positions)} positions × {len(rotations)} rotations = {site_grids.shape[0]} grids")

    # Create figure: positions as rows, rotations as columns
    n_rows = len(positions) * 2  # 2 rows per position (RGB + labels)
    n_cols = len(rotations) * 5  # 5 columns per rotation (RGB + 4 labels)

    fig = plt.figure(figsize=(5*len(rotations), 4*len(positions)))

    for pos_idx, position in enumerate(positions):
        for rot_idx, rotation in enumerate(rotations):
            # Find this specific grid
            grid_row = site_grids[
                (site_grids['position'] == position) &
                (site_grids['rotation'] == rotation)
            ]

            if grid_row.shape[0] == 0:
                continue

            zarr_idx = grid_row.iloc[0]['zarr_index']
            grid_id = grid_row.iloc[0]['grid_id']

            # Load data
            grid = pos_grids[zarr_idx]
            label_hard = pos_labels_hard[zarr_idx]
            label_sigma1 = pos_root['labels_soft_sigma1'][zarr_idx]
            label_sigma3 = pos_root['labels_soft_sigma3'][zarr_idx]
            label_sigma8 = pos_root['labels_soft_sigma8'][zarr_idx]

            # RGB
            r, g, b = grid[2], grid[1], grid[0]
            rgb = np.stack([r, g, b], axis=-1)
            p2, p98 = np.percentile(rgb, [2, 98])
            rgb_norm = np.clip((rgb - p2) / (p98 - p2), 0, 1)

            # Calculate subplot positions
            base_row = pos_idx * 2
            base_col = rot_idx * 5

            # Row 1: RGB + Hard label
            ax_rgb = plt.subplot2grid((n_rows, n_cols), (base_row, base_col), colspan=2)
            ax_rgb.imshow(rgb_norm)
            title = f'{position}\nRot {rotation}°'
            if rot_idx == 0:
                title = f'Position: {position.upper()}\nRot {rotation}°'
            ax_rgb.set_title(title, fontsize=10, fontweight='bold')
            ax_rgb.axis('off')

            ax_hard = plt.subplot2grid((n_rows, n_cols), (base_row, base_col+2))
            ax_hard.imshow(label_hard, cmap='gray', vmin=0, vmax=1)
            ax_hard.set_title(f'Hard\n{label_hard.sum()}px', fontsize=9)
            ax_hard.axis('off')

            # Row 2: Soft labels (sigma 1, 3, 8)
            ax_s1 = plt.subplot2grid((n_rows, n_cols), (base_row+1, base_col))
            ax_s1.imshow(label_sigma1, cmap='hot', vmin=0, vmax=1)
            ax_s1.set_title(f'σ=1', fontsize=9)
            ax_s1.axis('off')

            ax_s3 = plt.subplot2grid((n_rows, n_cols), (base_row+1, base_col+1))
            ax_s3.imshow(label_sigma3, cmap='hot', vmin=0, vmax=1)
            ax_s3.set_title(f'σ=3', fontsize=9)
            ax_s3.axis('off')

            ax_s8 = plt.subplot2grid((n_rows, n_cols), (base_row+1, base_col+2))
            ax_s8.imshow(label_sigma8, cmap='hot', vmin=0, vmax=1)
            ax_s8.set_title(f'σ=8', fontsize=9)
            ax_s8.axis('off')

    # Get site metadata
    site_info = site_grids.iloc[0]
    fig.suptitle(f'Site {site_index+1:06d} - All Augmentation Variants\n'
                 f'Site ID: {site_info["site_id"]}, Type: {site_info["site_type"]}\n'
                 f'Location: ({site_info["lat"]:.4f}, {site_info["lon"]:.4f})',
                 fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()


def visualize_random_samples(n_samples=3):
    """
    Visualize random samples from each category.

    Args:
        n_samples: Number of random samples per category
    """
    print("="*70)
    print("RANDOM SAMPLES VISUALIZATION")
    print("="*70)

    # Random positives
    print(f"\n{n_samples} Random Positive Samples:")
    print("-"*70)
    for i in range(n_samples):
        idx = np.random.randint(0, pos_grids.shape[0])
        visualize_sample_with_labels(idx, 'positive')

    # Random integrated negatives
    if ineg_grids.shape[0] > 0:
        print(f"\n{n_samples} Random Integrated Negative Samples:")
        print("-"*70)
        for i in range(n_samples):
            idx = np.random.randint(0, ineg_grids.shape[0])
            visualize_sample_with_labels(idx, 'integrated')

    # Random landcover negatives
    if lneg_grids.shape[0] > 0:
        print(f"\n{n_samples} Random Landcover Negative Samples:")
        print("-"*70)
        for i in range(n_samples):
            idx = np.random.randint(0, lneg_grids.shape[0])
            visualize_sample_with_labels(idx, 'landcover')

In [ ]:
# Visualization results
print("="*70)
print("VISUALIZATION TOOLS LOADED")
print("="*70)
print("\nAvailable functions:")
print("  1. visualize_sample_with_labels(zarr_index, sample_type)")
print("       - View one sample with all its labels")
print("       - sample_type: 'positive', 'integrated', or 'landcover'")
print("\n  2. visualize_all_variants_for_site(site_index)")
print("       - View ALL variants for one site")
print("       - Shows: all positions × all rotations × all labels")
print("       - This shows the complete augmentation pipeline!")
print("\n  3. visualize_random_samples(n_samples)")
print("       - View random samples from each category")
print("\nExamples:")
print("  visualize_sample_with_labels(0, 'positive')")
print("  visualize_all_variants_for_site(0)  # See everything for site 0!")
print("  visualize_random_samples(2)")
print("="*70)

# Pick a random site and show everything
num_unique_sites = pos_df['site_id'].nunique()
random_site_idx = np.random.randint(0, num_unique_sites)

print("\n" + "="*70)
print(f"COMPLETE AUGMENTATION PIPELINE - RANDOM SITE (Index {random_site_idx})")
print("="*70)
print("This shows ALL variants for ONE randomly selected site:")
print(f"  - All positions (center + random offsets)")
print(f"  - All rotations (0°, 90°, 180°, 270° or as configured)")
print(f"  - All labels (Hard + Soft σ=1,3,8)")
print(f"\nTotal grids shown: positions × rotations × 5 views")
print("="*70)

visualize_all_variants_for_site(random_site_idx)

print("\n" + "="*70)
print("VERIFICATION COMPLETE")
print("="*70)
print(f"✓ Displayed complete pipeline for site index {random_site_idx}")
print(f"✓ If this looks correct, your entire dataset is properly generated!")
print(f"\nTo see a different random site, re-run this cell.")
print(f"To see a specific site: visualize_all_variants_for_site(site_index)")
print("="*70)